# 00 · Earth Engine smoke test — Sentinel-5P HCHO + MAIAC AOD over India

**BAH 2026 PS3 · Phase-0 proof-of-life (Objective-1 & Objective-2 ingest backbone).**

This notebook authenticates against **Google Earth Engine (GEE)**, pulls **one**
Sentinel-5P TROPOMI **HCHO** tile and **one** MODIS **MAIAC AOD** tile over India,
and renders them. It exercises the real `aqi_india.ingest` adapters on the GEE path
(`COPERNICUS/S5P/OFFL/L3_HCHO` and `MODIS/061/MCD19A2_GRANULES`).

> ## ⚠️ REQUIRES EARTH ENGINE CREDENTIALS — DOES NOT RUN OFFLINE
>
> Unlike notebooks 01–06, this one **cannot** run on the offline synthetic path. It needs:
> - the `gee` extra installed: `pip install 'aqi_india[gee]'` (brings in `earthengine-api`, `geemap`), and
> - valid GEE credentials — either a **service-account** key (set `EE_SERVICE_ACCOUNT` and
>   `EE_SERVICE_ACCOUNT_KEY`) **or** cached user creds from `earthengine authenticate`,
>   plus a cloud project (`EARTHENGINE_PROJECT` / `GOOGLE_CLOUD_PROJECT`).
>
> If you only want to see the pipeline run, start at **`01_eda_synthetic_india.ipynb`** —
> the entire Objective-1/Objective-2 chain works offline on synthetic data with no credentials.

In [ ]:
import sys, pathlib
# Make the src/ layout importable when running from the notebooks/ folder
# without an editable install. If aqi_india is already installed this is a no-op.
_repo = pathlib.Path.cwd()
for _ in range(4):
    if (_repo / 'src' / 'aqi_india').is_dir():
        sys.path.insert(0, str(_repo / 'src'))
        break
    _repo = _repo.parent
import aqi_india
print('aqi_india', aqi_india.__version__)

## 1. Authenticate Earth Engine

`aqi_india.ingest.gee_auth.init_ee()` is idempotent and tries, in order: an explicit
service-account key, cached user credentials, then Application Default Credentials.
Set `EARTHENGINE_PROJECT` (or pass `project=`) to your GCP/EE cloud project.

In [ ]:
from aqi_india.ingest.gee_auth import init_ee, is_initialised

# Pass project=... explicitly if EARTHENGINE_PROJECT is not set in the environment.
init_ee()  # raises ImportError if earthengine-api is missing, RuntimeError if auth fails
print('Earth Engine initialised:', is_initialised())

## 2. Pull one Sentinel-5P TROPOMI HCHO tile over India

We use the real adapter `aqi_india.ingest.s5p.fetch_s5p` with `synthetic=False` so it hits
GEE. It filters `COPERNICUS/S5P/OFFL/L3_HCHO`, QA-masks with `qa_value >= 0.5`
(the blueprint floor in `S5P_QA_THRESHOLDS`), and median-composites the
`tropospheric_HCHO_column_number_density` band into a daily `hcho_col` cube
(`mol/m2`). For a quick smoke test we take a short window over the full India bbox.

In [ ]:
from aqi_india.ingest.s5p import (
    fetch_s5p, S5P_COLLECTIONS, S5P_BANDS, S5P_QA_THRESHOLDS,
)
from aqi_india.utils.geo import INDIA_BBOX

print('HCHO asset :', S5P_COLLECTIONS['hcho'])
print('HCHO band  :', S5P_BANDS['hcho'])
print('HCHO qa>=  :', S5P_QA_THRESHOLDS['hcho'])
print('India bbox :', INDIA_BBOX)

# One short window over India (the GEE path needs the 'xee' backend to read into xarray).
hcho = fetch_s5p('hcho', '2023-11-05', '2023-11-08', INDIA_BBOX, synthetic=False)
hcho

## 3. Pull one MAIAC 1 km AOD tile over India

`aqi_india.ingest.maiac.fetch_maiac` filters `MODIS/061/MCD19A2_GRANULES`, keeps
best-quality `Optical_Depth_055` pixels via the `AOD_QA` bitmask, scales by
`MAIAC_SCALE_FACTOR`, and median-composites the daily granules into an `aod` cube.

In [ ]:
from aqi_india.ingest.maiac import fetch_maiac, MAIAC_COLLECTION, MAIAC_BANDS

print('MAIAC collection:', MAIAC_COLLECTION)
print('MAIAC band      :', MAIAC_BANDS['aod_055'])

aod = fetch_maiac('2023-11-05', '2023-11-08', INDIA_BBOX, synthetic=False)
aod

## 4. Render the two tiles

A static matplotlib render of the first day of each cube (HCHO column + AOD) over India.
If you have `geemap` installed you can instead build an interactive split-map directly
from the GEE `ee.Image` objects — see the optional cell below.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
hcho['hcho_col'].isel(time=0).plot(ax=axes[0], cmap='magma', robust=True)
axes[0].set_title('S5P TROPOMI tropospheric HCHO column (mol/m2)')
aod['aod'].isel(time=0).plot(ax=axes[1], cmap='viridis', robust=True)
axes[1].set_title('MAIAC AOD 550 nm (unitless)')
for ax in axes:
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
fig.suptitle('Earth Engine smoke test — India', y=1.02)
plt.tight_layout()
plt.show()

### (Optional) Interactive geemap split-map straight from GEE

This bypasses the xarray export and renders the server-side `ee.Image` directly —
handy when the `xee` backend is unavailable. Requires `geemap`.

In [ ]:
# Optional — only runs if geemap is installed and EE is initialised.
try:
    import ee, geemap
    region = ee.Geometry.Rectangle(list(INDIA_BBOX))
    hcho_img = (ee.ImageCollection(S5P_COLLECTIONS['hcho'])
                .filterDate('2023-11-05', '2023-11-08').filterBounds(region)
                .select(S5P_BANDS['hcho']).median())
    aod_img = (ee.ImageCollection(MAIAC_COLLECTION)
               .filterDate('2023-11-05', '2023-11-08').filterBounds(region)
               .select(MAIAC_BANDS['aod_055']).median().multiply(0.001))
    m = geemap.Map(center=[23, 80], zoom=4)
    m.addLayer(hcho_img, {'min': 0, 'max': 3e-4, 'palette': ['black', 'red', 'yellow']}, 'S5P HCHO')
    m.addLayer(aod_img, {'min': 0, 'max': 1.5, 'palette': ['blue', 'green', 'red']}, 'MAIAC AOD')
    display(m)
except Exception as exc:
    print('geemap interactive map skipped:', type(exc).__name__, exc)

## Summary

If the cells above rendered an HCHO column and an AOD field over India, the GEE
connection, authentication, asset IDs and QA conventions are all wired correctly —
the Phase-0 gate. The remaining notebooks (01–06) run the **full pipeline offline** on
synthetic data via `aqi_india.sim`, so they need none of these credentials.